# 7.2 군집 평가 - 실루엣 분석 - 주석 설명판

원본 `7_2.ipynb`의 코드에 설명 주석을 단 버전이에요.

**왜 필요할까?**
KMeans는 군집 개수 K를 사람이 정해야 해요. 그런데 군집화는 정답이 없어서 정확도로 평가할 수 없어요.
→ **실루엣 계수(silhouette coefficient)** 로 '군집이 얼마나 잘 나뉘었는지'를 평가해요.

**실루엣 계수 계산 (데이터 1개마다)**
- **a** = 같은 군집 안의 다른 데이터들과의 평균 거리 (작을수록 좋음 → 우리 팀끼리 뭉쳐 있음)
- **b** = 가장 가까운 *다른* 군집의 데이터들과의 평균 거리 (클수록 좋음 → 옆 팀과 떨어져 있음)

$$s = \frac{b - a}{\max(a, b)}$$

| 값 | 의미 |
|---|---|
| 1에 가까움 | 자기 군집에 잘 들어가 있고 다른 군집과 멀리 떨어짐 👍 |
| 0 근처 | 두 군집의 경계에 걸쳐 있음 |
| 음수 | 다른 군집에 들어가는 게 더 맞음 (잘못 묶임) 👎 |

**좋은 군집화의 조건**
1. 전체 평균 실루엣 점수가 높다 (1에 가까움)
2. **군집별 실루엣 점수가 고르다** (한 군집만 높고 나머지는 낮으면 안 좋음)

## 1. 붓꽃 데이터 군집화 + 실루엣 계수 계산

In [ ]:
from sklearn.datasets import load_iris
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_samples, silhouette_score
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

iris = load_iris()
feature_names = ['sepal_length', 'sepal_width', 'petal_length', 'petal_width']
df = pd.DataFrame(data=iris.data, columns=feature_names)

# 3개 군집으로 KMeans 학습 (.fit을 바로 붙여서 만들기 + 학습을 한 줄로)
kmeans = KMeans(n_clusters=3, init='k-means++', max_iter=300, random_state=0).fit(df)
df['cluster'] = kmeans.labels_   # 각 데이터의 군집 번호

# silhouette_samples(X, 군집번호): 데이터 '하나하나'의 실루엣 계수
score_samples = silhouette_samples(iris.data, df['cluster'])
print('silhouette_samples의 리턴값의 모양', score_samples.shape)   # (150,) → 데이터 150개 각각의 값

df['silhouette_coeff'] = score_samples

# silhouette_score(X, 군집번호): 전체 데이터 실루엣 계수의 '평균' 하나
average_score = silhouette_score(iris.data, df['cluster'])
print('실루엣 점수', np.round(average_score, 3))   # 0.553

df.head()
# → 앞쪽 데이터(setosa)는 실루엣 계수가 0.8 이상으로 높음

## 2. 군집별 평균 실루엣 계수

In [ ]:
# 군집 번호별로 실루엣 계수의 평균 구하기
df.groupby('cluster')['silhouette_coeff'].mean()

# 결과:
#  군집 0 → 0.417
#  군집 1 → 0.798  ← setosa (7_1에서 완벽하게 분리됐던 군집)
#  군집 2 → 0.451
# → 전체 평균은 0.553이지만, 군집 1만 높고 0과 2는 낮음
#   = versicolor와 virginica가 서로 경계에 걸쳐 섞여 있다는 뜻 (7_1 결과와 같은 이야기)

## 3. 실루엣 분석 시각화 함수

군집 개수를 여러 개로 바꿔 가면서, 군집별 실루엣 계수를 **칼날 모양 그래프**로 비교하는 함수예요.

- 칼날 하나 = 군집 하나
- 칼날의 **두께** = 군집에 속한 데이터 수
- 칼날의 **길이** = 실루엣 계수 (오른쪽으로 길수록 좋음)
- 빨간 점선 = 전체 평균 실루엣 점수

In [ ]:
def visualize_silhouette(cluster_lists, X_features):
    import matplotlib.cm as cm   # 색상표(colormap)

    # 군집 개수 후보 수만큼 그래프를 가로로 배치
    n_cols = len(cluster_lists)
    fig, axs = plt.subplots(figsize=(4 * n_cols, 4), nrows=1, ncols=n_cols)

    # ind: 몇 번째 그래프인지 / n_cluster: 이번에 시험할 군집 개수
    for ind, n_cluster in enumerate(cluster_lists):

        # 군집 개수 n_cluster로 KMeans 학습
        clusterer = KMeans(n_clusters=n_cluster, max_iter=500, random_state=0)
        cluster_labels = clusterer.fit_predict(X_features)

        sil_avg = silhouette_score(X_features, cluster_labels)      # 전체 평균
        sil_values = silhouette_samples(X_features, cluster_labels) # 데이터별 값

        y_lower = 10   # 첫 번째 칼날이 시작할 y 위치 (아래 여백 10)

        # 제목: 군집 개수 + 평균 실루엣 점수
        axs[ind].set_title('No. of Cluster : ' + str(n_cluster) + '\n'
                           'Silhouette Score : ' + str(round(sil_avg, 3)))
        axs[ind].set_xlabel('The silhouette coefficient values')
        axs[ind].set_ylabel('Cluster Label')

        # x축: -0.1 ~ 1 (원본은 -.01 → 아래 군집 번호(x=-0.05)가 잘려서 -0.1로 수정)
        axs[ind].set_xlim([-0.1, 1])
        # y축: 데이터 수 + 칼날 사이 여백(10씩)만큼
        axs[ind].set_ylim([0, len(X_features) + (n_cluster + 1) * 10])
        axs[ind].set_yticks([])                              # y축 눈금 숨기기
        axs[ind].set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1])

        # 군집 하나씩 칼날 그리기
        for i in range(n_cluster):
            # i번째 군집에 속한 데이터의 실루엣 계수만 꺼내서 정렬
            ith_cluster_sil_values = sil_values[cluster_labels == i]
            ith_cluster_sil_values.sort()   # 작은 값 → 큰 값 순 (칼날 모양이 됨)

            size_cluster_i = ith_cluster_sil_values.shape[0]   # 군집의 데이터 수
            y_upper = y_lower + size_cluster_i                 # 칼날이 끝나는 y 위치

            color = cm.nipy_spectral(float(i) / n_cluster)     # 군집마다 다른 색

            # fill_betweenx(y범위, x시작, x끝): y 구간마다 x=0부터 실루엣 값까지 가로로 색칠
            axs[ind].fill_betweenx(np.arange(y_lower, y_upper), 0,
                                   ith_cluster_sil_values,
                                   facecolor=color, edgecolor=color, alpha=0.7)

            # 칼날 왼쪽 가운데에 군집 번호 표시
            axs[ind].text(-0.05, y_lower + 0.5 * size_cluster_i, str(i))

            y_lower = y_upper + 10   # 다음 칼날은 10만큼 띄워서 시작

        # 전체 평균 실루엣 점수 위치에 빨간 점선
        axs[ind].axvline(x=sil_avg, color='red', linestyle='--')

## 4. 가상 데이터로 최적의 군집 개수 찾기

In [ ]:
from sklearn.datasets import make_blobs

# 덩어리 4개짜리 가상 데이터 500개 → 정답은 '군집 4개'
X, y = make_blobs(n_samples=500, n_features=2, centers=4, cluster_std=1,
                  center_box=(-10.0, 10.0),   # 덩어리 중심이 -10~10 사이에 생김
                  shuffle=True, random_state=1)

# 군집 개수 2, 3, 4, 5개로 각각 실루엣 분석
visualize_silhouette([2, 3, 4, 5], X)

## 5. 결과 해석

| 군집 수 | 평균 실루엣 점수 | 군집별 상태 |
|---|---|---|
| 2 | **0.705** (가장 높음) | 375개짜리 두꺼운 칼날 + 125개짜리 칼날 → 덩어리 3개가 한 군집에 뭉쳐 있음 |
| 3 | 0.588 | 251개짜리 군집의 점수가 0.43으로 낮음, 음수인 데이터도 생김 |
| **4** | 0.651 | **칼날 4개의 두께(약 125개씩)와 길이가 고름**, 모두 빨간 점선 근처까지 옴 ✅ |
| 5 | 0.575 | 70개 이하짜리 작은 군집들의 점수가 0.3대로 낮음 → 덩어리 하나를 억지로 쪼갬 |

**⚠ 핵심 포인트**
평균 점수만 보면 **군집 2개**가 가장 높아요. 하지만 칼날 모양을 보면 군집 하나가 너무 두껍고 덩어리 여러 개를 억지로 묶은 상태예요.
→ **평균 점수가 높고 + 군집별로 고른** 군집 4개가 가장 좋은 선택이에요. (실제로 4개 덩어리로 만든 데이터)

그래서 실루엣 '점수'만이 아니라 **실루엣 '그래프'까지 같이 봐야** 해요.

In [ ]:
# (참고) 붓꽃 데이터에도 같은 분석을 해 볼 수 있어요
# 주석을 풀고 실행해 보세요
# visualize_silhouette([2, 3, 4, 5], iris.data)